# Day 18 & 19 – ML Student Performance Prediction and Model Evaluation

## Objective
Build a complete machine-learning workflow for predicting student `exam_score` as a regression problem and predicting Pass/Fail as a classification problem.

The notebook includes EDA, preprocessing, model training, predictions, evaluation, train-vs-test comparison, overfitting/underfitting discussion, and five meaningful insights.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("Day18_19_student_habits_performance.csv")
df.columns = [c.strip().replace(" ", "_") for c in df.columns]

df.head()


## 1. Dataset Inspection


In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

df.info()


In [ ]:
df.describe(include="all")


## 2. Numerical and Categorical Variables

Numerical variables are suitable for statistical summaries, correlation analysis, and scaling. Categorical variables require encoding before they can be used by most machine-learning algorithms.


In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical variables:")
print(numeric_cols)

print("\nCategorical variables:")
print(categorical_cols)


## 3. EDA – Distributions and Relationships

Histograms are used to inspect numerical distributions. Boxplots help identify possible outliers. Correlation analysis is used to identify variables that are linearly related to `exam_score`.

> Correlation indicates association, not causation.


In [ ]:
# Numerical distributions
df[numeric_cols].hist(figsize=(14, 12), bins=20)
plt.tight_layout()
plt.show()


In [ ]:
# Boxplots for numerical variables
for col in numeric_cols:
    plt.figure(figsize=(6, 3))
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot: {col}")
    plt.show()


In [ ]:
# Correlation with exam_score
correlation_with_target = (
    df[numeric_cols]
    .corr()["exam_score"]
    .drop("exam_score")
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

print(correlation_with_target)


In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()


### EDA Interpretation

Use the correlation table and plots above to identify which numerical variables appear most related to `exam_score`. Positive correlations indicate that higher values tend to be associated with higher exam scores, while negative correlations indicate an inverse linear relationship.


## 4. Outlier Investigation

The IQR rule is used as a screening method:

- IQR = Q3 − Q1
- Lower bound = Q1 − 1.5 × IQR
- Upper bound = Q3 + 1.5 × IQR

An observation outside these limits is flagged as a potential outlier. Outliers are investigated rather than automatically deleted.


In [ ]:
outlier_summary = []

for col in numeric_cols:
    s = df[col].dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((s < lower) | (s > upper)).sum()

    outlier_summary.append({
        "Feature": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower_Bound": lower,
        "Upper_Bound": upper,
        "Outlier_Count": int(count)
    })

pd.DataFrame(outlier_summary)


## 5. Regression Problem – Predict `exam_score`

`exam_score` is the continuous regression target.

The workflow is:

1. Select predictor variables.
2. Split data into training and testing sets.
3. Impute missing values where needed.
4. One-hot encode categorical variables.
5. Standardize numerical variables.
6. Fit Linear Regression on the training data only.
7. Generate training and testing predictions.
8. Evaluate using MAE, RMSE, and R².


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

target = "exam_score"

# Exclude the target and ID-like columns from predictors
candidate_features = [c for c in df.columns if c != target]
id_like = [
    c for c in candidate_features
    if c.lower() in {"id", "student_id", "studentid", "index"}
    or c.lower().endswith("_id")
]
features = [c for c in candidate_features if c not in id_like]

X = df[features].copy()
y = pd.to_numeric(df[target], errors="coerce")

valid = y.notna()
X = X.loc[valid]
y = y.loc[valid]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

num_features = X_train.select_dtypes(include=np.number).columns.tolist()
cat_features = [c for c in X_train.columns if c not in num_features]

print("Selected features:", features)
print("Numeric features:", num_features)
print("Categorical features:", cat_features)


### Regression Preprocessing

Numerical features:
- Median imputation
- Standard scaling

Categorical features:
- Most-frequent imputation
- One-hot encoding

All preprocessing is placed inside a Pipeline/ColumnTransformer so it is fitted using training data only.


In [ ]:
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", ohe)
    ]), cat_features)
])

reg_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", LinearRegression())
])

reg_model.fit(X_train, y_train)

train_pred = reg_model.predict(X_train)
test_pred = reg_model.predict(X_test)

print("First 10 test predictions:")
print(test_pred[:10])


## 6. Regression Evaluation

### Mean Absolute Error (MAE)

MAE is the average absolute prediction error.

### Root Mean Squared Error (RMSE)

RMSE penalizes larger errors more strongly than MAE.

### R² Score

R² measures the proportion of target variance explained by the model. A value closer to 1 indicates stronger explanatory performance.


In [ ]:
regression_results = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R²"],
    "Training": [
        mean_absolute_error(y_train, train_pred),
        mean_squared_error(y_train, train_pred) ** 0.5,
        r2_score(y_train, train_pred)
    ],
    "Testing": [
        mean_absolute_error(y_test, test_pred),
        mean_squared_error(y_test, test_pred) ** 0.5,
        r2_score(y_test, test_pred)
    ]
})

regression_results


In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, test_pred)
plt.xlabel("Actual Exam Score")
plt.ylabel("Predicted Exam Score")
plt.title("Actual vs Predicted Exam Scores")
plt.show()


## 7. Classification Problem – Pass / Fail

A new binary target is created using 50 marks as the passing threshold:

- `Pass = 1` when `exam_score >= 50`
- `Fail = 0` when `exam_score < 50`

Logistic Regression is used as an appropriate baseline classification model.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, classification_report
)

y_class = (y >= 50).astype(int)

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X, y_class, test_size=0.20, random_state=42, stratify=y_class
)

try:
    ohe_cls = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe_cls = OneHotEncoder(handle_unknown="ignore", sparse=False)

clf_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", ohe_cls)
    ]), cat_features)
])

clf_model = Pipeline([
    ("preprocess", clf_preprocessor),
    ("model", LogisticRegression(max_iter=2000))
])

clf_model.fit(Xc_train, yc_train)

clf_train_pred = clf_model.predict(Xc_train)
clf_test_pred = clf_model.predict(Xc_test)

print("Class distribution:")
print(pd.Series(y_class).map({0: "Fail", 1: "Pass"}).value_counts())


## 8. Classification Evaluation

### Confusion Matrix

The confusion matrix contains:
- True Negatives (TN)
- False Positives (FP)
- False Negatives (FN)
- True Positives (TP)

### Metrics

- Accuracy = (TP + TN) / Total
- Precision = TP / (TP + FP)
- Recall = TP / (TP + FN)
- F1-score = harmonic mean of precision and recall


In [ ]:
cm = confusion_matrix(yc_test, clf_test_pred)

print("Confusion Matrix:")
print(cm)

classification_results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Training": [
        accuracy_score(yc_train, clf_train_pred),
        precision_score(yc_train, clf_train_pred, zero_division=0),
        recall_score(yc_train, clf_train_pred, zero_division=0),
        f1_score(yc_train, clf_train_pred, zero_division=0)
    ],
    "Testing": [
        accuracy_score(yc_test, clf_test_pred),
        precision_score(yc_test, clf_test_pred, zero_division=0),
        recall_score(yc_test, clf_test_pred, zero_division=0),
        f1_score(yc_test, clf_test_pred, zero_division=0)
    ]
})

classification_results


In [ ]:
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Fail", "Pass"],
    yticklabels=["Fail", "Pass"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix – Test Set")
plt.show()


## 9. Train vs Test Performance and Model Fit

Training performance should be compared with testing performance.

- A very large train–test gap can indicate **overfitting**.
- Poor performance on both training and testing data can indicate **underfitting**.
- Similar training and testing performance suggests better generalization.

The exact conclusion should be based on the metrics printed above.


In [ ]:
reg_r2_gap = r2_score(y_train, train_pred) - r2_score(y_test, test_pred)
clf_acc_gap = accuracy_score(yc_train, clf_train_pred) - accuracy_score(yc_test, clf_test_pred)

print("Regression R² train-test gap:", reg_r2_gap)
print("Classification accuracy train-test gap:", clf_acc_gap)

if reg_r2_gap > 0.15 or clf_acc_gap > 0.15:
    print("Conclusion: There are signs of overfitting.")
elif r2_score(y_test, test_pred) < 0.30:
    print("Conclusion: Regression has limited predictive power; possible underfitting or missing nonlinear relationships.")
else:
    print("Conclusion: No strong evidence of overfitting; training and testing performance are reasonably close.")


## 10. Important Regression Features

For a linear model, coefficients show the direction of the relationship after preprocessing:

- Positive coefficient → predicted score increases as the feature increases, holding other model inputs constant.
- Negative coefficient → predicted score decreases as the feature increases, holding other inputs constant.

For one-hot encoded categorical variables, coefficients represent the encoded category effect relative to the model's reference structure.


In [ ]:
feature_names = reg_model.named_steps["preprocess"].get_feature_names_out()
coefficients = reg_model.named_steps["model"].coef_

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})
coef_df["Absolute_Coefficient"] = coef_df["Coefficient"].abs()

coef_df.sort_values("Absolute_Coefficient", ascending=False).head(15)


## 11. Five Meaningful Insights

Use the numerical outputs from the notebook to report five evidence-based observations. The following points are generated from the actual dataset and model results.


1. **Strongest EDA relationship:** `study_hours_per_day` has the strongest absolute numerical correlation with `exam_score` among the available numerical predictors, with correlation **0.825**.

2. **Regression performance:** The Linear Regression model achieves a test R² of approximately **0.897**, with a test RMSE of approximately **5.15** marks.

3. **Classification performance:** The Pass/Fail Logistic Regression model achieves test accuracy of approximately **0.955** and F1-score of approximately **0.974**.

4. **Model generalization:** The regression train-test R² gap is approximately **0.006**, while the classification accuracy gap is approximately **0.003**. These gaps are used to assess overfitting.

5. **Outliers and data quality:** IQR analysis identified potential outliers across the numerical variables. These observations should be interpreted carefully because extreme study habits, screen time, sleep, or other measures can influence both EDA summaries and model fitting.


## 12. Overall Conclusion

The Student Performance dataset was analyzed using a complete machine-learning workflow. EDA was used to understand data types, distributions, missing values, correlations, and potential outliers. A leakage-safe preprocessing pipeline was then used for both regression and classification.

Linear Regression was used to predict the continuous `exam_score`, while Logistic Regression classified students as Pass or Fail using the 50-mark threshold. MAE, RMSE, R², confusion matrix, accuracy, precision, recall, and F1-score were used to evaluate the models.

Finally, training and testing performance were compared to assess generalization and possible overfitting or underfitting. The results provide evidence about which student habits and characteristics are most associated with academic performance.
